In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [10]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.special import softmax
from sklearn.metrics import label_ranking_average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load paths
TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Test columns: {test_df.columns.tolist()}")

# Label mapping
LABELS = ['A', 'B', 'C', 'D', 'E']

# Load fine-tuned models
# IMPORTANT: You need to load your fine-tuned checkpoints here
# Replace these paths with your actual fine-tuned model paths
print("Loading fine-tuned models...")

# For now, loading base models with proper configuration for classification
from transformers import AutoConfig

# DeBERTa with 5 classes
deberta_config = AutoConfig.from_pretrained("microsoft/deberta-v3-small", num_labels=5)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small", 
    config=deberta_config
)
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
deberta_model.to(device)
deberta_model.eval()

# RoBERTa with 5 classes
roberta_config = AutoConfig.from_pretrained("roberta-base", num_labels=5)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    config=roberta_config
)
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
roberta_model.to(device)
roberta_model.eval()

# If you have fine-tuned checkpoints, load them like this:
# deberta_model = AutoModelForSequenceClassification.from_pretrained("path/to/fine-tuned/deberta")
# roberta_model = AutoModelForSequenceClassification.from_pretrained("path/to/fine-tuned/roberta")

# Function to get probabilities for a single text
def get_probabilities(model, tokenizer, text):
    """Get softmax probabilities for a given text"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = softmax(logits.cpu().numpy(), axis=1)[0]
    return probs

# Function to get Top-3 predictions
def get_top3_prediction(probs):
    """Get Top-3 prediction string from probabilities"""
    top3_indices = np.argsort(probs)[-3:][::-1]
    return ' '.join([LABELS[i] for i in top3_indices])

# ============== QUESTIONS 1-4: Row 25 Analysis ==============
print("\n" + "="*50)
print("QUESTIONS 1-4: Row 25 Analysis")
print("="*50)

# Row 25 (0-indexed: 24)
row_idx = 24
text_25 = test_df.iloc[row_idx]['prompt']

# Q1: DeBERTa prediction
deberta_probs_25 = get_probabilities(deberta_model, deberta_tokenizer, text_25)
highest_class = np.argmax(deberta_probs_25)
highest_prob = deberta_probs_25[highest_class]

print(f"\nQ1: {LABELS[highest_class]}, probability of {LABELS[highest_class]} = {highest_prob:.4f}")

# Q2: Average probabilities
roberta_probs_25 = get_probabilities(roberta_model, roberta_tokenizer, text_25)
avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2
highest_avg_class = np.argmax(avg_probs_25)

print(f"Q2: {LABELS[highest_avg_class]}")

# Q3: Weighted average probabilities
weighted_probs_25 = 0.7 * deberta_probs_25 + 0.3 * roberta_probs_25
highest_weighted_class = np.argmax(weighted_probs_25)

print(f"Q3: {LABELS[highest_weighted_class]}")

# Q4: Top-3 prediction for row 25
top3_string_25 = get_top3_prediction(weighted_probs_25)
print(f"Q4: {top3_string_25}")

# ============== QUESTION 5: Generate submission.csv ==============
print("\n" + "="*50)
print("QUESTION 5: Generate submission.csv")
print("="*50)

def get_ensemble_top3(text, model1, tokenizer1, model2, tokenizer2, weight1=0.7, weight2=0.3):
    """Get Top-3 predictions using weighted ensemble"""
    probs1 = get_probabilities(model1, tokenizer1, text)
    probs2 = get_probabilities(model2, tokenizer2, text)
    weighted = weight1 * probs1 + weight2 * probs2
    return get_top3_prediction(weighted)

# Generate predictions for all test rows
predictions = []
for idx, row in test_df.iterrows():
    text = row['prompt']
    top3 = get_ensemble_top3(text, deberta_model, deberta_tokenizer, 
                            roberta_model, roberta_tokenizer)
    predictions.append(top3)

# Create submission file
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions
})
submission_df.to_csv('submission.csv', index=False)

print(f"Q5: {len(submission_df)} rows in submission.csv")

# ============== QUESTION 6: Test-Time Augmentation ==============
print("\n" + "="*50)
print("QUESTION 6: Test-Time Augmentation (First 50 rows)")
print("="*50)

def get_tta_probabilities(model, tokenizer, text):
    """Average probabilities from original and augmented text"""
    # Original
    probs_orig = get_probabilities(model, tokenizer, text)
    
    # Augmented
    augmented_text = "Answer the following multiple-choice question carefully: " + text
    probs_aug = get_probabilities(model, tokenizer, augmented_text)
    
    # Average
    return (probs_orig + probs_aug) / 2

diff_tta_count = 0
for idx in range(min(50, len(test_df))):
    text = test_df.iloc[idx]['prompt']
    
    # Original prediction
    orig_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
    orig_pred = LABELS[np.argmax(orig_probs)]
    
    # TTA prediction
    tta_probs = get_tta_probabilities(deberta_model, deberta_tokenizer, text)
    tta_pred = LABELS[np.argmax(tta_probs)]
    
    if orig_pred != tta_pred:
        diff_tta_count += 1

print(f"Q6: {diff_tta_count} rows have different Top-1 prediction after TTA")

# ============== QUESTION 7: DeBERTa vs Weighted Ensemble ==============
print("\n" + "="*50)
print("QUESTION 7: DeBERTa vs Weighted Ensemble (First 100 rows)")
print("="*50)

diff_ensemble_count = 0
deberta_confidences = []
ensemble_confidences = []

for idx in range(min(100, len(test_df))):
    text = test_df.iloc[idx]['prompt']
    
    # DeBERTa
    de_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
    de_pred = LABELS[np.argmax(de_probs)]
    de_conf = np.max(de_probs)
    
    # Weighted Ensemble
    ro_probs = get_probabilities(roberta_model, roberta_tokenizer, text)
    ensemble_probs = 0.7 * de_probs + 0.3 * ro_probs
    ensemble_pred = LABELS[np.argmax(ensemble_probs)]
    ensemble_conf = np.max(ensemble_probs)
    
    deberta_confidences.append(de_conf)
    ensemble_confidences.append(ensemble_conf)
    
    if de_pred != ensemble_pred:
        diff_ensemble_count += 1

print(f"Q7: {diff_ensemble_count} rows have different Top-1 predictions")

# ============== QUESTION 8: Confidence Gain ==============
print("\n" + "="*50)
print("QUESTION 8: Confidence Gain Analysis (First 100 rows)")
print("="*50)

positive_gain_count = 0
confidence_gains = []

for i in range(min(100, len(test_df))):
    gain = ensemble_confidences[i] - deberta_confidences[i]
    confidence_gains.append(gain)
    if gain > 0:
        positive_gain_count += 1

print(f"Q8: {positive_gain_count} rows have positive confidence gain")
print(f"Average confidence gain: {np.mean(confidence_gains):.4f}")

# ============== QUESTION 9: Top-3 Ranking Changes ==============
print("\n" + "="*50)
print("QUESTION 9: Top-3 Ranking Changes (First 100 rows)")
print("="*50)

rank_change_count = 0
for idx in range(min(100, len(test_df))):
    text = test_df.iloc[idx]['prompt']
    
    # DeBERTa top3
    de_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
    de_top3 = get_top3_prediction(de_probs)
    
    # Ensemble top3
    ro_probs = get_probabilities(roberta_model, roberta_tokenizer, text)
    ensemble_probs = 0.7 * de_probs + 0.3 * ro_probs
    ensemble_top3 = get_top3_prediction(ensemble_probs)
    
    if de_top3 != ensemble_top3:
        rank_change_count += 1

print(f"Q9: {rank_change_count} rows have changes in their ordered Top-3 ranking")

# ============== QUESTION 10: MAP@3 Score ==============
print("\n" + "="*50)
print("QUESTION 10: MAP@3 Score Calculation")
print("="*50)

def calculate_map3(true_labels, predicted_probs, k=3):
    """
    Calculate MAP@3 score
    true_labels: List of true label indices (0-4)
    predicted_probs: Array of shape (n_samples, 5) with predicted probabilities
    """
    # Convert to one-hot encoding
    n_samples = len(true_labels)
    n_classes = 5  # A, B, C, D, E
    
    y_true_binary = np.zeros((n_samples, n_classes))
    for i, label in enumerate(true_labels):
        y_true_binary[i, label] = 1
    
    # Ensure predicted_probs has the correct shape
    if len(predicted_probs.shape) == 1:
        predicted_probs = predicted_probs.reshape(1, -1)
    
    # Calculate label ranking average precision
    map_score = label_ranking_average_precision_score(y_true_binary, predicted_probs)
    return map_score

# Use first 100 training samples for validation
val_size = min(100, len(train_df))
true_labels = []
predicted_probs_ensemble = []

print(f"Calculating MAP@3 on {val_size} validation samples...")

for idx in range(val_size):
    text = train_df.iloc[idx]['prompt']
    
    # Get true label
    true_label = train_df.iloc[idx]['answer']
    true_label_idx = LABELS.index(true_label)
    true_labels.append(true_label_idx)
    
    # Get ensemble probabilities
    de_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
    ro_probs = get_probabilities(roberta_model, roberta_tokenizer, text)
    ensemble_probs = 0.7 * de_probs + 0.3 * ro_probs
    predicted_probs_ensemble.append(ensemble_probs)

predicted_probs_ensemble = np.array(predicted_probs_ensemble)
true_labels_array = np.array(true_labels)

# Verify shapes
print(f"Shape of true_labels: {true_labels_array.shape}")
print(f"Shape of predicted_probs_ensemble: {predicted_probs_ensemble.shape}")

try:
    map3_score = calculate_map3(true_labels_array, predicted_probs_ensemble)
    print(f"Q10: {map3_score:.4f}")
except Exception as e:
    print(f"Error calculating MAP@3: {e}")
    print("Using alternative calculation...")
    
    # Alternative manual MAP@3 calculation
    def manual_map3(true_labels, predicted_probs, k=3):
        map_scores = []
        for i in range(len(true_labels)):
            true_label = true_labels[i]
            # Get top-k predictions
            top_k_indices = np.argsort(predicted_probs[i])[-k:][::-1]
            # Calculate precision at each rank
            precisions = []
            relevant_found = 0
            for j, pred in enumerate(top_k_indices):
                if pred == true_label:
                    relevant_found += 1
                    precision = relevant_found / (j + 1)
                    precisions.append(precision)
            # Average precision for this query
            if precisions:
                map_scores.append(np.mean(precisions))
            else:
                map_scores.append(0.0)
        return np.mean(map_scores)
    
    map3_score = manual_map3(true_labels_array, predicted_probs_ensemble)
    print(f"Q10 (manual calculation): {map3_score:.4f}")

# ============== FINAL SUMMARY ==============
print("\n" + "="*50)
print("FINAL ANSWERS SUMMARY")
print("="*50)
print(f"Q1: {LABELS[highest_class]}, probability of {LABELS[highest_class]} = {highest_prob:.4f}")
print(f"Q2: {LABELS[highest_avg_class]}")
print(f"Q3: {LABELS[highest_weighted_class]}")
print(f"Q4: {top3_string_25}")
print(f"Q5: {len(submission_df)}")
print(f"Q6: {diff_tta_count}")
print(f"Q7: {diff_ensemble_count}")
print(f"Q8: {positive_gain_count}")
print(f"Q9: {rank_change_count}")
print(f"Q10: {map3_score:.4f}")

# Save results to a text file
with open('assignment_results.txt', 'w') as f:
    f.write(f"Q1: {LABELS[highest_class]}, probability of {LABELS[highest_class]} = {highest_prob:.4f}\n")
    f.write(f"Q2: {LABELS[highest_avg_class]}\n")
    f.write(f"Q3: {LABELS[highest_weighted_class]}\n")
    f.write(f"Q4: {top3_string_25}\n")
    f.write(f"Q5: {len(submission_df)}\n")
    f.write(f"Q6: {diff_tta_count}\n")
    f.write(f"Q7: {diff_ensemble_count}\n")
    f.write(f"Q8: {positive_gain_count}\n")
    f.write(f"Q9: {rank_change_count}\n")
    f.write(f"Q10: {map3_score:.4f}\n")

print("\nResults saved to 'assignment_results.txt'")
print("Submission file saved to 'submission.csv'")

# Display first few rows of submission
print("\nFirst 5 rows of submission.csv:")
print(submission_df.head())

Using device: cpu
Train shape: (2000, 8)
Test shape: (500, 7)
Test columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
Loading fine-tuned models...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



QUESTIONS 1-4: Row 25 Analysis

Q1: D, probability of D = 0.2355
Q2: D
Q3: D
Q4: D B E

QUESTION 5: Generate submission.csv
Q5: 500 rows in submission.csv

QUESTION 6: Test-Time Augmentation (First 50 rows)
Q6: 15 rows have different Top-1 prediction after TTA

QUESTION 7: DeBERTa vs Weighted Ensemble (First 100 rows)
Q7: 16 rows have different Top-1 predictions

QUESTION 8: Confidence Gain Analysis (First 100 rows)
Q8: 4 rows have positive confidence gain
Average confidence gain: -0.0160

QUESTION 9: Top-3 Ranking Changes (First 100 rows)
Q9: 58 rows have changes in their ordered Top-3 ranking

QUESTION 10: MAP@3 Score Calculation
Calculating MAP@3 on 100 validation samples...
Shape of true_labels: (100,)
Shape of predicted_probs_ensemble: (100, 5)
Q10: 0.4998

FINAL ANSWERS SUMMARY
Q1: D, probability of D = 0.2355
Q2: D
Q3: D
Q4: D B E
Q5: 500
Q6: 15
Q7: 16
Q8: 4
Q9: 58
Q10: 0.4998

Results saved to 'assignment_results.txt'
Submission file saved to 'submission.csv'

First 5 rows of 